
# 02a_Metadaten_Features_Engineering

## Ziel
Dieses Notebook erzeugt **deterministische Features** aus den harmonisierten Metadaten (`metadata_base.parquet`).  
Es konzentriert sich auf Merkmale, die die Viralität eines Kurzvideos erklären oder beeinflussen könnten —  
ohne maschinelles Lernen, rein regelbasiert und reproduzierbar.

Das Ergebnis dient als Eingabe für das nächste Notebook  
📘 *02b_Metadaten_Features_Modellierung.ipynb* (Feature-Bewertung und -Selektion).

---

## Governance & Guardrails

- **Keine Leakage-Features:** Es werden keine Variablen verwendet, die nach dem Label-Zeitpunkt entstehen (z. B. finale `likes`, `views`, `comments`).  
- **Deterministisches Feature-Engineering:** Alle erzeugten Features sind direkt aus den Upload-Metadaten oder Beschreibungen ableitbar.  
- **Ziel:** Erklärbare Features, die später im Gesamtmodell (T4) fusioniert werden können.  
- **Speicherorte:**  
  - Zwischenergebnisse & Schema → `notebooks/`  
  - Endgültige Feature-Datei → `features/metadata_features.csv`


## 1. Setup & Import

In [ ]:

from pathlib import Path
import pandas as pd, numpy as np, re, json, warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path("..").resolve()
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
FEATURES_DIR = PROJECT_ROOT / "features"

pd.set_option("display.max_colwidth", 160)

base_path = NOTEBOOKS_DIR / "metadata_base.parquet"
assert base_path.exists(), f"{base_path} nicht gefunden. Bitte erst 01_Metadaten_Intake_und_Harmonisierung ausführen."
df = pd.read_parquet(base_path)
print("Geladene Zeilen:", len(df))
df.head(3)



## 2. Zeitbezogene Features
Erstellt Merkmale aus dem Upload-Zeitpunkt (Stunde, Wochentag, zirkuläre Kodierung).


In [ ]:

df["upload_time"] = pd.to_datetime(df.get("upload_time"), utc=True, errors="coerce")

df["hour"] = df["upload_time"].dt.hour
df["weekday"] = df["upload_time"].dt.weekday
df["upload_hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
df["upload_hour_cos"] = np.cos(2*np.pi*df["hour"]/24)
df["is_weekend"] = df["weekday"].isin([5,6]).astype(int)


## 3. Account-bezogene Features

In [ ]:

if "creator_verified" in df.columns:
    df["creator_verified"] = df["creator_verified"].fillna(False).astype(int)

for c in ["creator_follower_count", "creator_posts_count"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
        df[f"{c}_log"] = np.log1p(df[c])


## 4. Text- und Hashtag-Features

In [ ]:

def safe_str(x): return x if isinstance(x, str) else ""
cap = df["description"].map(safe_str) if "description" in df.columns else pd.Series([""]*len(df))

df["caption_len_chars"] = cap.str.len()
df["caption_len_words"] = cap.str.split().map(len)
df["has_link_or_mention"] = cap.str.contains(r"(https?://|www\.|@)", case=False).fillna(False).astype(int)
df["has_question_mark"] = cap.str.contains(r"\?", regex=True).fillna(False).astype(int)
df["has_exclamation"] = cap.str.contains(r"!", regex=True).fillna(False).astype(int)

def extract_tags(s):
    s = safe_str(s)
    return re.findall(r"#\w+", s.lower())

tags = cap.map(extract_tags)
df["hashtags_count"] = tags.map(len)
df["unique_hashtags_count"] = tags.map(lambda t: len(set(t)))


## 5. Video-Metadaten-Features

In [ ]:

if "duration_s" in df.columns:
    df["duration_s"] = pd.to_numeric(df["duration_s"], errors="coerce")
    df["duration_bucket"] = pd.cut(
        df["duration_s"], bins=[0, 15, 30, 60, 600, np.inf],
        labels=["very_short", "short", "mid", "long", "very_long"]
    )


## 6. Schema, Qualitätskontrolle & Export

In [ ]:

feature_cols = [
    "video_id", "is_viral",
    "creator_verified",
    "creator_follower_count_log", "creator_posts_count_log",
    "upload_hour_sin", "upload_hour_cos", "weekday", "is_weekend",
    "duration_s", "duration_bucket",
    "caption_len_chars", "caption_len_words",
    "has_link_or_mention", "has_question_mark", "has_exclamation",
    "hashtags_count", "unique_hashtags_count"
]

feature_cols = [c for c in feature_cols if c in df.columns]
feat = df[feature_cols].copy()

raw_path = NOTEBOOKS_DIR / "metadata_features_raw.csv"
feat.to_csv(raw_path, index=False)
print("Roh-Feature-Datei gespeichert:", raw_path)

schema = {c: str(feat[c].dtype) for c in feat.columns}
schema_path = NOTEBOOKS_DIR / "schema_metadata_features.json"
schema_path.write_text(json.dumps(schema, indent=2, ensure_ascii=False), encoding="utf-8")
print("Schema gespeichert:", schema_path)

display(feat.head(5))



---
✅ **Ergebnis:**
- `metadata_features_raw.csv` (deterministische Feature-Tabelle)  
- `schema_metadata_features.json` (Spaltentypen und Struktur)

Nächster Schritt:  
📘 *02b_Metadaten_Features_Modellierung.ipynb* — Analyse der Feature-Bedeutung und Export der finalen `metadata_features.csv`.
